# q14v5 probe — why is a 0.611 member reading 0.167?

v57 refused q14v5 because it measured **0.1673** on 1,500 pool rows. The refusal was right — the gate
kept the score at 0.627 — but the number is impossible: the same adapter is the **3.0-weight member**
of the 0.654/0.660 public blend, whose solo score is **0.611**.

The first run of this probe died on `a leaf Variable that requires grad is being used in an in-place
operation` — `merge_lora` was missing its `@torch.no_grad()`. That is a bug in this probe only; the
production cell has the decorator, and v57's merge completed cleanly, so the 0.1673 was measured with
a properly merged adapter.

This runs it on 600 rows it was **trained on**, where accuracy must be high if the loader is right:

| variant | final norm | eos |
|---|---|---|
| 1 | removed (what v57 did) | 151643 |
| 2 | kept | 151643 |
| 3 | removed | tokenizer's |
| 4 | kept | tokenizer's |

One variant jumping to ~0.9 names the fix. All four near random means the mismatch is elsewhere
(layer list, head assembly) and the next probe narrows that. No submission.

In [ ]:
"""Why did q14v5 read 0.1673 on pool rows when its public score is 0.611?

The gate did its job and refused the member, so nothing was lost — but a member that scores 0.611 on
the leaderboard cannot be at random accuracy on a labelled pool unless our inference path differs from
the path it was trained with. This probes the three things that could differ, on rows the adapter was
actually trained on (train.csv), where accuracy must be high if the loader is right:

    variant 1  final norm removed   eos 151643      <- what v57 did
    variant 2  final norm kept      eos 151643
    variant 3  final norm removed   eos = tokenizer's
    variant 4  final norm kept      eos = tokenizer's

If every variant is near random the loader is wrong in a way none of these explain (layer list, head
assembly). If one variant jumps, that is the fix.
"""
import glob
import json
import os
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from safetensors.torch import load_file
from transformers import AutoConfig, AutoModel, AutoTokenizer

T0 = time.time()
INPUT = "/kaggle/input"
LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
          'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
          'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5',
          'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
L2I = {l: i for i, l in enumerate(LABELS)}
KEEP, LAYERS, MAXLEN, N_ROWS = 27, [2, 4, 6, 8, 12, 16, 20, 24], 384, 600


def find(pattern):
    hits = glob.glob(f"{INPUT}/**/{pattern}", recursive=True)
    return hits[0] if hits else None


def find_base(spec):
    for cfg in sorted(glob.glob(f"{INPUT}/**/config.json", recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if all(c.get(k) == v for k, v in spec.items()):
            return os.path.dirname(cfg)
    return None


def load_trunk(base_dir, keep, dtype, keep_norm):
    mc = AutoConfig.from_pretrained(base_dir)
    mc.num_hidden_layers, mc.use_cache = keep, False
    if getattr(mc, "layer_types", None) is not None:
        mc.layer_types = mc.layer_types[:keep]
    model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map="auto",
                                      max_memory={i: "13GiB" for i in range(torch.cuda.device_count())}).eval()
    if not keep_norm:
        model.norm = nn.Identity()
    return model


@torch.no_grad()
def merge_lora(model, adapter_dir):
    acfg = json.load(open(os.path.join(adapter_dir, "adapter_config.json")))
    scale = acfg["lora_alpha"] / acfg["r"]
    sd = load_file(os.path.join(adapter_dir, "adapter_model.safetensors"))
    modules = dict(model.named_modules())
    hit = 0
    for k, A in sd.items():
        if ".lora_A." not in k:
            continue
        name = k.split(".lora_A.")[0]
        try:
            lin = modules[name[name.index("layers."):]]
        except Exception:
            continue
        B = sd[k.replace(".lora_A.", ".lora_B.")]
        lin.weight.add_(((B.float().to(lin.weight.device) @ A.float().to(lin.weight.device)) * scale).to(lin.weight.dtype))
        hit += 1
    print(f"  merged {hit} LoRA pairs (of {sum(1 for k in sd if '.lora_A.' in k)})", flush=True)


def features(model, tok, heads, texts, eos_id, bs=8):
    H = model.config.hidden_size
    norms = nn.ModuleList([nn.LayerNorm(H) for _ in LAYERS]).to("cuda:0").float()
    norms.load_state_dict({k[len("norms."):]: v for k, v in heads.items() if k.startswith("norms.")})
    ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=MAXLEN)["input_ids"]
    ids = [x + [eos_id] for x in ids]
    order = np.argsort([len(x) for x in ids], kind="stable")
    Z = np.zeros((len(ids), H * len(LAYERS)), np.float32)
    with torch.no_grad():
        for s in range(0, len(order), bs):
            idx = order[s:s + bs]
            enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(model.embed_tokens.weight.device)
            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
            m = enc["attention_mask"].unsqueeze(-1).float().to("cuda:0")
            m[:, 0] = 0
            den = m.sum(1).clamp(min=1)
            Z[idx] = torch.cat([nm((hs[l].to("cuda:0").float() * m).sum(1) / den)
                                for l, nm in zip(LAYERS, norms)], -1).cpu().numpy()
    return Z

## Load the adapter, the base and 600 in-sample rows

In [ ]:
adir = None
for c in glob.glob(f"{INPUT}/**/q14v5/adapter_config.json", recursive=True):
    adir = os.path.dirname(c)
    break
print("adapter dir:", adir, flush=True)
assert adir, "attach dhruvtibarewal/nlm-l24-adapters-v5"
heads = torch.load(os.path.join(adir, "heads.pt"), map_location="cpu", weights_only=False)
base_dir = find_base({"hidden_size": 5120, "model_type": "qwen3"})
print("base dir:", base_dir, flush=True)

train = pd.read_csv(find("train.csv"))
rng = np.random.default_rng(0)
idx = rng.choice(len(train), size=min(N_ROWS, len(train)), replace=False)
texts = train.text.astype(str).values[idx].tolist()
y = train.model.map(L2I).values[idx]
tok0 = AutoTokenizer.from_pretrained(base_dir)
print(f"{len(texts)} in-sample train rows | tokenizer eos {tok0.eos_token_id}", flush=True)

## Four variants

In [ ]:
results = {}
for keep_norm in (False, True):
    dtype = torch.bfloat16
    model = load_trunk(base_dir, KEEP, dtype, keep_norm)
    merge_lora(model, adir)
    tok = AutoTokenizer.from_pretrained(base_dir)
    tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    for eos in (151643, tok.eos_token_id):
        key = f"norm={'kept' if keep_norm else 'removed'} eos={eos}"
        try:
            Z = features(model, tok, heads, texts, eos)
            P = torch.softmax(torch.tensor(Z) @ heads["plain.weight"].T + heads["plain.bias"], -1).numpy()
            acc = float((P.argmax(1) == y).mean())
            results[key] = round(acc, 4)
            print(f"{key}: in-sample accuracy {acc:.4f}", flush=True)
        except Exception as e:
            results[key] = f"failed: {type(e).__name__}: {e}"
            print(f"{key}: FAILED {type(e).__name__}: {e}", flush=True)
    del model
    torch.cuda.empty_cache()

json.dump(results, open("q14_probe.json", "w"), indent=2)
print("\n==== summary ====", flush=True)
for k, v in results.items():
    print(f"  {k}: {v}", flush=True)
print(f"\ndone in {time.time() - T0:.0f}s", flush=True)